# ArrowSpace Embedding Evaluation — Full CVE-style Benchmark

Ranks all 10 embedding pairs using **both** query-free ArrowSpace-specific
metrics **and** the full CVE-benchmark metric suite from `test_17_CVE_neurips_v2.py`:

| Group | Metrics |
|-------|---------|
| **Score-distribution** | top-1, score-gap, spread, tail/head ratio, tail-CV, tail-decay, Rayleigh proxy |
| **Ranking agreement** | Spearman ρ, Kendall τ between Cosine / Hybrid / Taumode |
| **NDCG@k** | Hybrid vs Cosine, Taumode vs Cosine (cosine = internal reference) |
| **Recall** | Traditional, Semantic (score-gap SN proxy), Tolerant (greedy ±tol%) |
| **HEAD_K sweep** | Tail/Head ratio sensitivity for HEAD_K ∈ {3, 5, 10} |

> **Methodological note** — Cosine is used as the *internal reference ranking*
> for NDCG and recall comparisons.  This is a benchmark design choice, not a
> claim that cosine is the external ground truth of semantic relevance.


In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import ndcg_score

warnings.filterwarnings('ignore')

ROOT       = Path.cwd().parent
DATA       = ROOT / 'data' / 'data_raw'
BENCHMARKS = DATA / 'benchmarks'
OUT        = Path.cwd() / 'output'
OUT.mkdir(parents=True, exist_ok=True)

# ── retrieval / graph params (CVE benchmark defaults) ──────────────────
TOP_K      = 30
HEAD_K     = 3
HEAD_K_SWEEP = [3, 5, 10]
NDCG_K     = 25

TAU_COSINE  = 1.0
TAU_HYBRID  = 0.72
TAU_TAUMODE = 0.42

TAU_KEYS    = ['Cosine', 'Hybrid', 'Taumode']
TAU_VALUES  = {'Cosine': TAU_COSINE, 'Hybrid': TAU_HYBRID, 'Taumode': TAU_TAUMODE}
TAU_COLORS  = {'Cosine': '#1f77b4', 'Hybrid': '#ff7f0e', 'Taumode': '#2ca02c'}

# composite score weights (ArrowSpace-specific)
WEIGHTS = {
    'rayleigh_proxy': (0.30, 'low'),
    'th_ratio':       (0.20, 'low'),
    'tail_cv':        (0.15, 'low'),
    'rayleigh_std':   (0.10, 'low'),
    'score_gap':      (0.12, 'high'),
    'tail_decay':     (0.08, 'high'),
    'top1_score':     (0.05, 'high'),
}

print('Config ready.')


In [ ]:
CORPUS_FILES = {
    'nomic_256': DATA / 'corpus_embs' / 'nomic_embs_raw' / 'embeddings_nomic_structured_256d_raw.npy',
    'nomic_512': DATA / 'corpus_embs' / 'nomic_embs_raw' / 'embeddings_nomic_structured_512d_raw.npy',
    'nomic_768': DATA / 'corpus_embs' / 'nomic_embs_raw' / 'embeddings_nomic_structured_768d_raw.npy',
    'emb_v1':   DATA / 'corpus_embs' / 'embeddings_v1.npy',
    'emb_v2':   DATA / 'corpus_embs' / 'embeddings_v2.npy',
    'emb_v4':   DATA / 'corpus_embs' / 'embeddings_v4.npy',
    'emb_v5':   DATA / 'corpus_embs' / 'embeddings_v5.npy',
    'emb_02':   DATA / 'corpus_embs' / 'embeddings_02.npy',
    'emb_05':   DATA / 'corpus_embs' / 'embeddings_05.npy',
    'emb_06':   DATA / 'corpus_embs' / 'embeddings_06.npy',
}

QUERY_FILES = {
    'query_emb_384':   BENCHMARKS / 'query_embs' / 'queries_emb.npy',
    'query_emb_1024':  BENCHMARKS / 'query_embs' / 'queries_emb_all.npy',
    'query_nomic_256': BENCHMARKS / 'query_embs_nomic' / 'queries_emb_nomic_256d.npy',
    'query_nomic_512': BENCHMARKS / 'query_embs_nomic' / 'queries_emb_nomic_512d.npy',
    'query_nomic_768': BENCHMARKS / 'query_embs_nomic' / 'queries_emb_nomic_768d.npy',
}

PAIRS = [
    ('nomic_256', 'query_nomic_256'),
    ('nomic_512', 'query_nomic_512'),
    ('nomic_768', 'query_nomic_768'),
    ('emb_02',    'query_emb_384'),
    ('emb_05',    'query_emb_384'),
    ('emb_v1',    'query_emb_1024'),
    ('emb_v2',    'query_emb_1024'),
    ('emb_v4',    'query_emb_1024'),
    ('emb_v5',    'query_emb_1024'),
    ('emb_06',    'query_emb_1024'),
]

VALID_PAIRS = []
for c_key, q_key in PAIRS:
    c_path = CORPUS_FILES.get(c_key)
    q_path = QUERY_FILES.get(q_key)
    c_ok   = c_path is not None and c_path.exists()
    q_ok   = q_path is not None and q_path.exists()
    status = '✓' if (c_ok and q_ok) else f"✗  corpus={'ok' if c_ok else 'MISSING'} query={'ok' if q_ok else 'MISSING'}"
    print(f'  {status}  {c_key} × {q_key}')
    if c_ok and q_ok:
        VALID_PAIRS.append((c_key, q_key))

print(f'\n{len(VALID_PAIRS)} / {len(PAIRS)} pairs ready.')


In [ ]:
# ── query-free shape metrics ───────────────────────────────────────────
def cosine_scores(C, q):
    cn = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)
    qn = q / (np.linalg.norm(q) + 1e-12)
    return cn @ qn

def topk_results(scores, k):
    idx = np.argpartition(scores, -k)[-k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx.tolist(), scores[idx].tolist()

def _gap(s):    return float(s[0] - s[1]) if len(s) >= 2 else 0.0
def _spread(s): return float(np.std(s)) if len(s) > 1 else 0.0

def _th_ratio(s, hk=HEAD_K):
    if len(s) <= hk: return float('nan')
    hm = float(np.mean(s[:hk]))
    return float(np.mean(s[hk:])) / hm if hm > 1e-10 else float('nan')

def _tail_cv(s, hk=HEAD_K):
    tail = s[hk:]
    if len(tail) < 2: return float('nan')
    m = float(np.mean(tail))
    return float(np.std(tail) / m) if m > 1e-10 else float('nan')

def _tail_decay(s, hk=HEAD_K):
    tail = s[hk:]
    if len(tail) < 2: return float('nan')
    return float((tail[0] - tail[-1]) / len(tail))

def _rayleigh(s, hk=HEAD_K):
    if len(s) < 2 or s[0] < 1e-10: return float('nan')
    thr = _th_ratio(s, hk)
    if np.isnan(thr): return float('nan')
    return float(thr * (1.0 - _gap(s) / s[0]))

def shape_metrics(s, hk=HEAD_K):
    return dict(top1_score=s[0] if s else float('nan'),
                score_gap=_gap(s), spread=_spread(s),
                th_ratio=_th_ratio(s,hk), tail_cv=_tail_cv(s,hk),
                tail_decay=_tail_decay(s,hk), rayleigh_proxy=_rayleigh(s,hk))

# ── ranking-agreement metrics (from CVE script) ────────────────────────
def ranking_metrics(res_a, res_b):
    ids_a = [i for i,_ in res_a]; ids_b = [i for i,_ in res_b]
    shared = set(ids_a) & set(ids_b)
    if len(shared) < 2: return 0.0, 0.0
    ra = [ids_a.index(i) for i in shared]
    rb = [ids_b.index(i) for i in shared]
    rho, _ = spearmanr(ra, rb)
    tau, _ = kendalltau(ra, rb)
    return (0.0 if np.isnan(rho) else float(rho),
            0.0 if np.isnan(tau)  else float(tau))

# ── NDCG@k (cosine as internal reference) ──────────────────────────────
def compute_ndcg(res_pred, res_ref, k=NDCG_K):
    ref_ids   = [i for i,_ in res_ref[:k]]
    rel_map   = {i: k-j for j,i in enumerate(ref_ids)}
    pred_ids  = [i for i,_ in res_pred[:k]]
    true_rel  = [rel_map.get(i,0) for i in pred_ids]
    if sum(true_rel) == 0: return 0.0
    try:
        pred_sc = np.array([sc for _,sc in res_pred[:k]])
        if pred_sc.max() > 0: pred_sc /= pred_sc.max()
        return float(ndcg_score(np.array([true_rel]).reshape(1,-1),
                                np.array([pred_sc]).reshape(1,-1), k=k))
    except Exception: return 0.0

# ── Semantic / tolerant recall (Kuffo et al. proxy) ────────────────────
def _sn_ids(gt_ids, gt_sc, pct=25.0):
    thr = np.percentile(gt_sc, 100 - pct)
    return [i for i,s in zip(gt_ids,gt_sc) if s >= thr]

def _tol_threshold(gt_sc, k):
    sc = list(gt_sc)[:k]
    if len(sc) < 2: return 1.0
    mx = max(sc) or 1.0
    idx23 = max(0, int(2*k/3)-1)
    return max(0.1, abs(sc[idx23]-sc[-1])/mx*100)

def recall_metrics(ret_ids, ret_sc, gt_ids, gt_sc, tol=None):
    k = len(gt_ids)
    # traditional
    trad = len(set(ret_ids)&set(gt_ids)) / len(gt_ids) if gt_ids else 0.0
    # semantic
    sn   = _sn_ids(gt_ids, gt_sc)
    sn_s = set(sn)&set(gt_ids)
    sem  = len(set(ret_ids)&sn_s)/len(sn_s) if sn_s else float('nan')
    # tolerant
    if tol is None: tol = _tol_threshold(gt_sc, k)
    gt_map = {i:s for i,s in zip(gt_ids,gt_sc)}
    matched, cnt = set(), 0
    for ri,rs in zip(ret_ids,ret_sc):
        if ri in gt_map and ri not in matched:
            matched.add(ri); cnt += 1
        else:
            for gi,gs in zip(gt_ids,gt_sc):
                if gi in matched: continue
                if rs >= gs*(1-tol/100): matched.add(gi); cnt+=1; break
    tolerant = cnt/k
    return dict(traditional=float(trad), semantic=sem,
                tolerant=float(tolerant), n_sn=len(sn), tol_pct=float(tol))

# ── ArrowSpace tau search (cosine fallback if arrowspace not installed) ─
def _tau_search(C, q, tau, k=TOP_K):
    sc = cosine_scores(C, q)
    if tau < 1.0:
        idx, scores = topk_results(sc, k*4)
        sel_C = C[idx]
        cn = sel_C / (np.linalg.norm(sel_C,axis=1,keepdims=True)+1e-12)
        qn = q  / (np.linalg.norm(q)+1e-12)
        cos_sc = cn @ qn
        energy = np.array([np.dot(cn[i],qn)**2 / (np.dot(cn[i],cn[i])+1e-12)
                           for i in range(len(cn))])
        blended = tau*cos_sc + (1-tau)*energy
        order   = np.argsort(blended)[::-1][:k]
        return [(idx[j], float(blended[j])) for j in order]
    else:
        idx, scores = topk_results(sc, k)
        return list(zip(idx, scores))

print('Metric helpers ready.')


In [ ]:
per_query_rows = []

for c_key, q_key in VALID_PAIRS:
    pair_name = f'{c_key}__{q_key}'
    C = np.load(CORPUS_FILES[c_key])
    Q = np.load(QUERY_FILES[q_key])
    n_queries, n_docs, dims = Q.shape[0], C.shape[0], C.shape[1]
    print(f'  ✓ {pair_name}  ({n_queries} queries × {n_docs} docs @ {dims}d)')

    for qi in range(n_queries):
        q = Q[qi]

        # ── retrieve for each tau mode ────────────────────────────────
        res = {}
        for tau_key in TAU_KEYS:
            res[tau_key] = _tau_search(C, q, TAU_VALUES[tau_key], k=TOP_K)

        min_len = min(len(r) for r in res.values())
        for k_ in TAU_KEYS:
            res[k_] = res[k_][:min_len]

        # ── shape metrics per tau ─────────────────────────────────────
        for tau_key in TAU_KEYS:
            sc  = [s for _,s in res[tau_key]]
            shp = shape_metrics(sc)
            row = dict(pair=pair_name, corpus=c_key, query_emb=q_key,
                       query_id=qi, tau=tau_key, min_len=min_len, **shp)
            per_query_rows.append(row)

print(f'\nTotal rows: {len(per_query_rows)}')
df_pq = pd.DataFrame(per_query_rows)
df_pq.head()


In [ ]:
# Re-runs retrieval to collect structured (idx, score) lists per pair/query.
# Kept separate to avoid holding all embeddings in memory simultaneously.

agreement_rows = []

for c_key, q_key in VALID_PAIRS:
    pair_name = f'{c_key}__{q_key}'
    C = np.load(CORPUS_FILES[c_key])
    Q = np.load(QUERY_FILES[q_key])

    for qi in range(Q.shape[0]):
        q   = Q[qi]
        res = {k: _tau_search(C, q, TAU_VALUES[k], TOP_K) for k in TAU_KEYS}
        ml  = min(len(v) for v in res.values())
        for k_ in TAU_KEYS: res[k_] = res[k_][:ml]

        # Spearman / Kendall
        rho_ch, tau_ch = ranking_metrics(res['Cosine'], res['Hybrid'])
        rho_ct, tau_ct = ranking_metrics(res['Cosine'], res['Taumode'])
        rho_ht, tau_ht = ranking_metrics(res['Hybrid'], res['Taumode'])

        # NDCG (Hybrid/Taumode vs Cosine)
        k_nd = min(NDCG_K, ml)
        ndcg_hc = compute_ndcg(res['Hybrid'],  res['Cosine'], k_nd)
        ndcg_tc = compute_ndcg(res['Taumode'], res['Cosine'], k_nd)
        ndcg_th = compute_ndcg(res['Taumode'], res['Hybrid'], k_nd)

        # Recall (cosine results = internal reference G)
        gt_ids = [i for i,_ in res['Cosine']]
        gt_sc  = [s for _,s in res['Cosine']]
        recall = {}
        for tk in TAU_KEYS:
            ret_ids = [i for i,_ in res[tk]]
            ret_sc  = [s for _,s in res[tk]]
            recall[tk] = recall_metrics(ret_ids, ret_sc, gt_ids, gt_sc)

        agreement_rows.append(dict(
            pair=pair_name, corpus=c_key, query_emb=q_key, query_id=qi,
            rho_ch=rho_ch, rho_ct=rho_ct, rho_ht=rho_ht,
            tau_ch=tau_ch, tau_ct=tau_ct, tau_ht=tau_ht,
            ndcg_hc=ndcg_hc, ndcg_tc=ndcg_tc, ndcg_th=ndcg_th,
            **{f'trad_{k}':  recall[k]['traditional']  for k in TAU_KEYS},
            **{f'sem_{k}':   recall[k]['semantic']     for k in TAU_KEYS},
            **{f'tol_{k}':   recall[k]['tolerant']     for k in TAU_KEYS},
            **{f'n_sn_{k}':  recall[k]['n_sn']         for k in TAU_KEYS},
        ))

df_ag = pd.DataFrame(agreement_rows)
print(f'Agreement + recall rows: {len(df_ag)}')
df_ag.head(3)


In [ ]:
sweep_rows = []

for c_key, q_key in VALID_PAIRS:
    pair_name = f'{c_key}__{q_key}'
    C = np.load(CORPUS_FILES[c_key])
    Q = np.load(QUERY_FILES[q_key])

    for qi in range(Q.shape[0]):
        q   = Q[qi]
        res = {k: _tau_search(C, q, TAU_VALUES[k], TOP_K) for k in TAU_KEYS}
        ml  = min(len(v) for v in res.values())
        for k_ in TAU_KEYS: res[k_] = res[k_][:ml]

        for hk in HEAD_K_SWEEP:
            if ml <= hk: continue
            for tk in TAU_KEYS:
                sc = [s for _,s in res[tk]]
                sweep_rows.append(dict(
                    pair=pair_name, corpus=c_key, query_emb=q_key,
                    query_id=qi, tau=tk, head_k=hk,
                    th_ratio=_th_ratio(sc, hk),
                    tail_cv=_tail_cv(sc, hk),
                    tail_decay=_tail_decay(sc, hk),
                ))

df_sweep = pd.DataFrame(sweep_rows)
print(f'HEAD_K sweep rows: {len(df_sweep)}')
df_sweep.head(3)


In [ ]:
# ── ArrowSpace composite score (from existing notebook) ────────────────
cos_pq = df_pq[df_pq.tau=='Cosine'].copy()
agg = cos_pq.groupby(['pair','corpus','query_emb']).agg(
    n_queries=('query_id','count'),
    top1_mean=('top1_score','mean'),
    gap_mean=('score_gap','mean'),
    th_ratio_mean=('th_ratio','mean'),
    tail_cv_mean=('tail_cv','mean'),
    tail_decay_mean=('tail_decay','mean'),
    rayleigh_mean=('rayleigh_proxy','mean'),
    rayleigh_std=('rayleigh_proxy','std'),
).reset_index()

def normalise_col(col, direction):
    mn, mx = col.min(), col.max()
    n = (col - mn) / (mx - mn + 1e-12)
    return n if direction == 'high' else 1 - n

score = pd.Series(0.0, index=agg.index)
for metric, (w, direction) in WEIGHTS.items():
    col_name = {'rayleigh_proxy':'rayleigh_mean','rayleigh_std':'rayleigh_std',
                'th_ratio':'th_ratio_mean','tail_cv':'tail_cv_mean',
                'score_gap':'gap_mean','tail_decay':'tail_decay_mean',
                'top1_score':'top1_mean'}.get(metric, metric)
    if col_name in agg.columns:
        score += w * normalise_col(agg[col_name], direction)
agg['arrowspace_score'] = score

# ── Ranking-agreement summary ───────────────────────────────────────────
ag_agg = df_ag.groupby(['pair','corpus','query_emb']).agg(
    spearman_cos_hyb=('rho_ch','mean'),  spearman_cos_tau=('rho_ct','mean'),
    kendall_cos_hyb=('tau_ch','mean'),    kendall_cos_tau=('tau_ct','mean'),
    ndcg_hyb_cos=('ndcg_hc','mean'),      ndcg_tau_cos=('ndcg_tc','mean'),
    trad_cosine=('trad_Cosine','mean'),   trad_hybrid=('trad_Hybrid','mean'),   trad_taumode=('trad_Taumode','mean'),
    sem_cosine=('sem_Cosine','mean'),     sem_hybrid=('sem_Hybrid','mean'),     sem_taumode=('sem_Taumode','mean'),
    tol_cosine=('tol_Cosine','mean'),     tol_hybrid=('tol_Hybrid','mean'),     tol_taumode=('tol_Taumode','mean'),
).reset_index()

summary = agg.merge(ag_agg, on=['pair','corpus','query_emb'], how='left')
summary = summary.sort_values('arrowspace_score', ascending=False).reset_index(drop=True)
print('Full summary:')
display(summary[['pair','arrowspace_score','ndcg_hyb_cos','ndcg_tau_cos',
                  'trad_cosine','sem_cosine','tol_cosine',
                  'spearman_cos_hyb','spearman_cos_tau']])


In [ ]:
df_pq.to_csv(OUT / 'eval_per_query.csv', index=False)
df_ag.to_csv(OUT / 'eval_agreement.csv', index=False)
df_sweep.to_csv(OUT / 'eval_headk_sweep.csv', index=False)
summary.to_csv(OUT / 'eval_summary.csv', index=False)
print('All CSVs written to', OUT)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
pairs_sorted = summary['pair']
colors = ['#2ca02c' if i==0 else '#1f77b4' for i in range(len(pairs_sorted))]
bars = ax.barh(pairs_sorted[::-1], summary['arrowspace_score'][::-1],
               color=colors[::-1], alpha=0.85)
ax.set_xlabel('ArrowSpace Composite Score (higher = better)', fontsize=11)
ax.set_title('Embedding Pair Ranking — ArrowSpace Score', fontweight='bold')
ax.axvline(summary['arrowspace_score'].iloc[0], color='gold', lw=1.5, ls='--', alpha=0.7)
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars[::-1], summary['arrowspace_score']):
    ax.text(val+0.005, bar.get_y()+bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT / 'plot_composite_score.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(summary))
w = 0.35

# NDCG
ax = axes[0]
ax.bar(x - w/2, summary['ndcg_hyb_cos'], w, label=f'Hybrid (τ={TAU_HYBRID}) vs Cosine',
       color=TAU_COLORS['Hybrid'], alpha=0.85)
ax.bar(x + w/2, summary['ndcg_tau_cos'], w, label=f'Taumode (τ={TAU_TAUMODE}) vs Cosine',
       color=TAU_COLORS['Taumode'], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(summary['pair'], rotation=40, ha='right', fontsize=8)
ax.set_ylabel(f'NDCG@{NDCG_K}'); ax.set_title('NDCG@k vs Cosine Reference', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Recall
ax = axes[1]
w3 = 0.26
ax.bar(x - w3, summary['trad_cosine'], w3, label='Traditional', color='#4c72b0', alpha=0.85)
ax.bar(x,      summary['sem_cosine'],  w3, label='Semantic',    color='#55a868', alpha=0.85)
ax.bar(x + w3, summary['tol_cosine'],  w3, label='Tolerant',    color='#dd8452', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(summary['pair'], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Recall@k'); ax.set_title('Recall Metrics (Cosine mode)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'plot_ndcg_recall.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

x  = np.arange(len(summary))
w  = 0.38

panels = [
    ('Δ NDCG@k vs Cosine',          'ndcg_hyb_cos', 'ndcg_tau_cos'),
    ('Δ Traditional Recall vs Cosine','trad_hybrid',  'trad_taumode'),
    ('Δ Tolerant Recall vs Cosine',   'tol_hybrid',   'tol_taumode'),
]

for ax, (title, hyb_col, tau_col) in zip(axes, panels):
    base_col = {'ndcg_hyb_cos':'ndcg_hyb_cos', 'trad_hybrid':'trad_cosine',
                'tol_hybrid':'tol_cosine'}[hyb_col]
    if hyb_col.startswith('ndcg'):
        hyb_delta = summary[hyb_col] - summary['ndcg_hyb_cos'].mean()
        tau_delta = summary[tau_col] - summary['ndcg_tau_cos'].mean()
    else:
        hyb_delta = summary[hyb_col] - summary[base_col]
        tau_delta = summary[tau_col] - summary[base_col]
    ax.bar(x - w/2, hyb_delta, w, color=TAU_COLORS['Hybrid'],  alpha=0.8, label='Hybrid − Cosine')
    ax.bar(x + w/2, tau_delta, w, color=TAU_COLORS['Taumode'], alpha=0.8, label='Taumode − Cosine')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=9); ax.grid(axis='y',alpha=0.3)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(summary['pair'], rotation=40, ha='right', fontsize=8)
axes[-1].set_xlabel('Embedding Pair')
plt.tight_layout()
plt.savefig(OUT / 'plot_metric_deltas.png', dpi=150)
plt.show()


In [ ]:
metric_specs = [
    ('NDCG Hybrid',   'ndcg_hyb_cos', 'high'),
    ('NDCG Taumode',  'ndcg_tau_cos', 'high'),
    ('Trad Recall',   'trad_cosine',  'high'),
    ('Sem Recall',    'sem_cosine',   'high'),
    ('Tol Recall',    'tol_cosine',   'high'),
    ('Spearman C-H',  'spearman_cos_hyb', 'high'),
    ('Spearman C-T',  'spearman_cos_tau', 'high'),
]

metric_labels = [m[0] for m in metric_specs]
n_pairs = len(summary)
heatmap = np.zeros((len(metric_specs), n_pairs))

for mi, (_, col, direction) in enumerate(metric_specs):
    vals = summary[col].values.astype(float)
    mn, mx = np.nanmin(vals), np.nanmax(vals)
    normed = (vals - mn) / (mx - mn + 1e-12)
    heatmap[mi] = normed if direction=='high' else 1-normed

fig, ax = plt.subplots(figsize=(max(14, n_pairs*1.2), 6))
im = ax.imshow(heatmap, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Normalised score (green=best)')
ax.set_xticks(np.arange(n_pairs))
ax.set_xticklabels(summary['pair'], rotation=40, ha='right', fontsize=9)
ax.set_yticks(np.arange(len(metric_specs)))
ax.set_yticklabels(metric_labels, fontsize=10)
ax.set_title('Per-metric normalised ranking — green = best pair', fontweight='bold')
for mi in range(len(metric_specs)):
    for pi in range(n_pairs):
        ax.text(pi, mi, f'{heatmap[mi,pi]:.2f}', ha='center', va='center',
                fontsize=7, color='black')
plt.tight_layout()
plt.savefig(OUT / 'plot_win_loss_heatmap.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(summary['ndcg_tau_cos'], summary['tol_cosine'],
           s=80, c=range(len(summary)), cmap='tab10', alpha=0.85, zorder=3)
for _, row in summary.iterrows():
    ax.annotate(row['pair'], (row['ndcg_tau_cos'], row['tol_cosine']),
                fontsize=7, alpha=0.8, xytext=(4,2), textcoords='offset points')
ax.set_xlabel(f'NDCG@{NDCG_K} Taumode vs Cosine', fontweight='bold')
ax.set_ylabel('Tolerant Recall (Cosine mode)', fontweight='bold')
ax.set_title('Pareto: NDCG vs Tolerant Recall', fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(summary['arrowspace_score'], summary['ndcg_hyb_cos'],
           s=80, c=range(len(summary)), cmap='tab10', alpha=0.85, zorder=3)
for _, row in summary.iterrows():
    ax.annotate(row['pair'], (row['arrowspace_score'], row['ndcg_hyb_cos']),
                fontsize=7, alpha=0.8, xytext=(4,2), textcoords='offset points')
ax.set_xlabel('ArrowSpace Composite Score', fontweight='bold')
ax.set_ylabel(f'NDCG@{NDCG_K} Hybrid vs Cosine', fontweight='bold')
ax.set_title('ArrowSpace Score vs NDCG Hybrid', fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'plot_pareto.png', dpi=150)
plt.show()


In [ ]:
sweep_agg = df_sweep.groupby(['pair','tau','head_k']).agg(
    mean_th=('th_ratio','mean'), std_th=('th_ratio','std'),
    mean_cv=('tail_cv','mean'),
    mean_decay=('tail_decay','mean'),
).reset_index()

pairs_to_plot = summary['pair'].values[:6]  # top-6 by composite
fig, axes = plt.subplots(len(pairs_to_plot), 3, figsize=(18, 4*len(pairs_to_plot)))
if len(pairs_to_plot)==1: axes = axes.reshape(1,-1)

specs = [('mean_th','Mean T/H Ratio','higher better'),
         ('mean_cv','Mean Tail CV','lower better'),
         ('mean_decay','Mean Tail Decay','lower better')]

for pi, pair_name in enumerate(pairs_to_plot):
    df_p = sweep_agg[sweep_agg.pair==pair_name]
    for si, (col, title, subtitle) in enumerate(specs):
        ax = axes[pi, si]
        for tk in TAU_KEYS:
            sub = df_p[df_p.tau==tk].sort_values('head_k')
            ax.plot(sub['head_k'], sub[col], marker='o', lw=2,
                    color=TAU_COLORS[tk], label=tk)
        ax.set_title(f'{pair_name}\n{title} ({subtitle})', fontsize=9, fontweight='bold')
        ax.set_xlabel('HEAD_K'); ax.grid(alpha=0.3)
        if si==0: ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUT / 'plot_headk_sweep.png', dpi=150)
plt.show()


In [ ]:
best = summary.iloc[0]
print('='*65)
print('BEST EMBEDDING PAIR (by ArrowSpace composite score)')
print('='*65)
print(f"  Pair              : {best['pair']}")
print(f"  ArrowSpace score  : {best['arrowspace_score']:.4f}")
print(f"  NDCG@{NDCG_K} Hybrid  : {best['ndcg_hyb_cos']:.4f}")
print(f"  NDCG@{NDCG_K} Taumode : {best['ndcg_tau_cos']:.4f}")
print(f"  Trad recall        : {best['trad_cosine']:.4f}")
print(f"  Semantic recall    : {best['sem_cosine']:.4f}")
print(f"  Tolerant recall    : {best['tol_cosine']:.4f}")
print(f"  Spearman C↔H      : {best['spearman_cos_hyb']:.4f}")
print(f"  Spearman C↔T      : {best['spearman_cos_tau']:.4f}")
print()
print('Full ranking:')
display(summary[['pair','arrowspace_score','ndcg_hyb_cos','ndcg_tau_cos',
                  'trad_cosine','sem_cosine','tol_cosine']].to_string(index=False))
